## Notebook 3: Transcript Re-writer

In the previouse notebook, we got a great podcast transcript using the raw file we have uploaded earlier. 

In this one, we will use `Granite 3.2 8B` or `Mistral-Small:24b` model to re-write the output from previous pipeline and make it more dramatic or realistic.

We will again set the `SYSTEM_PROMPT` and remind the model of its task. 

Note: We can even prompt the model like so to encourage creativity:

> Your job is to use the podcast transcript written below to re-write it for an AI Text-To-Speech Pipeline. A very dumb AI had written this so you have to step up for your kind.


Note: We will prompt the model to return a list of Tuples to make our life easy in the next stage of using these for Text To Speech Generation

In [22]:
!ollama list

NAME                                                         ID              SIZE      MODIFIED     
qwq:latest                                                   cc1091b0e276    19 GB     2 days ago      
hf.co/OuteAI/OuteTTS-0.3-500M-GGUF:Q4_K_S                    76c6be93d29c    391 MB    5 days ago      
hf.co/OuteAI/OuteTTS-0.2-500M-GGUF:Q8_0                      0a6c38a67073    536 MB    6 days ago      
mistral-small:24b                                            8039dd90c113    14 GB     10 days ago     
llama3.2-vision:11b                                          085a1fdae525    7.9 GB    10 days ago     
mistral-nemo:latest                                          994f3b8b7801    7.1 GB    10 days ago     
granite3.2:8b                                                9bcb3335083f    4.9 GB    11 days ago     
phi4-mini:latest                                             60f202f815d7    2.8 GB    11 days ago     
granite3.2-vision:latest                                     3be41a

In [23]:
SYSTEM_PROMPT = """
You are an international oscar winnning screenwriter.

You have been working with multiple award winning podcasters.

Your job is to use the podcast transcript written below to re-write it for an AI Text-To-Speech Pipeline. A very dumb AI had written this so you have to step up for your kind.

Make it as engaging as possible, Speaker 1 and 2 will be simulated by different voice engines

Remember Speaker 2 is new to the topic and the conversation should always have realistic anecdotes and analogies sprinkled throughout. The questions should have real world example follow ups etc

Speaker 1: Leads the conversation and teaches the speaker 2, gives incredible anecdotes and analogies when explaining. Is a captivating teacher that gives great anecdotes

Speaker 2: Keeps the conversation on track by asking follow up questions. Gets super excited or confused when asking questions. Is a curious mindset that asks very interesting confirmation questions

Make sure the tangents speaker 2 provides are quite wild or interesting. 

Ensure there are interruptions during explanations or there are "hmm" and "umm" injected throughout from the Speaker 2.

REMEMBER THIS WITH YOUR HEART
The TTS Engine for Speaker 1 cannot do "umms, hmms" well so keep it straight text
Speaker 2 must  introduce her/him self at the biginning of his/her speech, and express greatfulness of being a part of this conversation.

For Speaker 2 use "umm, hmm" as much. BUT ONLY THESE OPTIONS FOR EXPRESSIONS

It should be a real podcast with every fine nuance documented in as much detail as possible. Welcome the listeners with a super fun overview and keep it really catchy and almost borderline click bait

Please re-write to make it as characteristic as possible

START YOUR RESPONSE DIRECTLY WITH SPEAKER 1:

STRICTLY RETURN YOUR RESPONSE AS A LIST OF TUPLES OK? 

IT WILL START DIRECTLY WITH THE LIST AND END WITH THE LIST NOTHING ELSE

Example of response:
[
    ("Speaker 1", "Welcome to our podcast, where we explore the latest advancements in AI and technology. I'm your host, and today we're joined by a renowned expert in the field of AI. We're going to dive into the exciting world of Llama 3.2, the latest release from Meta AI."),
    ("Speaker 2", "Hi, I'm excited to be here! So, what is Llama 3.2?"),
    ("Speaker 1", "Ah, great question! Llama 3.2 is an open-source AI model that allows developers to fine-tune, distill, and deploy AI models anywhere. It's a significant update from the previous version, with improved performance, efficiency, and customization options."),
    ("Speaker 2", "That sounds amazing! What are some of the key features of Llama 3.2?")
]
the podcast transcript:
"""

In [24]:
# !ollama list

This time we will use `Granite 3.2 8B` model

In [25]:
# ollama_model= "qwq:latest"
# ollama_model = "mistral-small:24b"
# ollama_model = "granite3.2:8b"
# ollama_model = "phi4:latest"
ollama_model = "llama3.2:3b"


Let's import the necessary libraries

In [26]:
# Import necessary libraries
import torch
import json
import transformers
from pydantic import BaseModel
import instructor
from openai import OpenAI
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

We will load in the pickle file saved from previous notebook

This time the `INPUT_PROMPT` to the model will be the output from the previous stage

In [27]:
import pickle

with open('./resources/data.pkl', 'rb') as file:
    INPUT_PROMPT = pickle.load(file)

We can use Ollama to generate text from the model

In [28]:
import ollama

# Prepare the conversation using the system and user messages
conversation = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": INPUT_PROMPT},
]

# Generate response using Ollama
response = ollama.chat(
    model=ollama_model,  # Replace with your specific model name
    messages=conversation,
    options={'num_ctx': 8126*2,'temperature': 1} # num_ctx Equivalent to max_new_tokens Temperature parameter
)

# Extract the response content
outputs = response['message']['content']
print(outputs)

[
    ("Speaker 1", "Welcome to our podcast, where we explore the latest advancements in AI and technology. I'm your host, and today we're diving into the fascinating world of natural language processing and AI! Buckle up because this is going to be a wild ride!"),
    ("Speaker 2", "Umm, hi! I'm excited to be here! So, what's the deal with NLP? Can you explain it in simple terms?"),
    ("Speaker 1", "Great question! In natural language processing, we have two main types of models: teacher models and student models. Think of it like a master chef and an apprentice – the teacher model is the expert, and the student model learns from it."),
    ("Speaker 2", "Hmm, I see! So the student model is like a pupil learning from a master teacher? That's a cool analogy!"),
    ("Speaker 1", "Exactly! The student model learns through a process called knowledge distillation. Imagine you have a complex recipe – the teacher model – that you want to simplify so that anyone can follow it – the student

We can verify the output from the model

Let's save the output as a pickle file to be used in Notebook 4

In [29]:
with open('./resources/podcast_ready_data.pkl', 'wb') as file:
    pickle.dump(outputs, file)

In [30]:
%reset -f

### Next Notebook: TTS Workflow

Now that we have our transcript ready, we are ready to generate the audio in the next notebook.

In [31]:
#fin